<a href="https://colab.research.google.com/github/zencolab/colab/blob/main/comfyui_audio0429.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# Cell 1: 部署 ComfyUI 底座与系统级音频库
# ==========================================
import os

print("🧹 1. 正在彻底清空旧环境，防止残留干扰...")
!rm -rf /content/ComfyUI

print("⏳ 2. 安装系统级底层音频依赖 (FFmpeg & libsndfile1)...")
!apt update -qq
!apt -y install ffmpeg libsndfile1 -qq

print("🔄 3. 下载全新 ComfyUI 官方源码...")
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git

print("📦 4. 安装核心底座依赖...")
%cd /content/ComfyUI
!pip install -q -r requirements.txt
print("✅ Cell 1 完成！底座极其纯净。")

🧹 1. 正在彻底清空旧环境，防止残留干扰...
⏳ 2. 安装系统级底层音频依赖 (FFmpeg & libsndfile1)...
111 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
libsndfile1 is already the newest version (1.0.31-2ubuntu0.2).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 111 not upgraded.
🔄 3. 下载全新 ComfyUI 官方源码...
/content
Cloning into 'ComfyUI'...
remote: Enumerating objects: 36884, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 36884 (delta 69), reused 40 (delta 40), pack-reused 36790 (from 3)
Receiving objects: 100% (36884/36884), 79.43 MiB | 17.44 MiB/s, done.
Resolving deltas: 100% (24720/24720), done.
📦 4. 安装核心底座依赖...
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.9/21.

In [ ]:
# ==========================================
# Cell 2: 安装 FishAudio 官方节点与环境强力修复
# ==========================================
import os
%cd /content/ComfyUI/custom_nodes

print("⏳ [1/3] 克隆 FishAudioS2 及配套节点仓库...")
# 官方指定的节点源码
!git clone https://github.com/Saganaki22/ComfyUI-FishAudioS2.git
# 必装：用来合成带声音的 MP4 视频
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git

print("📦 [2/3] 正在静默安装官方 requirements 依赖...")
!pip install -q -r ComfyUI-FishAudioS2/requirements.txt
!pip install -q -r ComfyUI-VideoHelperSuite/requirements.txt

print("💉 [3/3] 注入终极环境稳定剂 (极其关键)...")
# 1. 强力升级 bitsandbytes 适配 CUDA 12.8，彻底防止底层编译报错
# 2. 提前打入 descript-audiotools 等包，彻底阻断首次运行时的超时 Fetch 崩溃
!pip install -q -U "bitsandbytes>=0.45.0" descript-audiotools descript-audio-codec
# 3. 降级 numpy 解决兼容性，并补齐底层音频解码包
!pip install -q "numpy<2.0.0" resampy soundfile librosa pydub huggingface_hub
# 4. 补充 SageAttention 库，解决 Voice Clone 节点的注意力机制报错问题
!pip install -q sageattention

print("\n============================================================")
print("✅ Cell 2 完成！环境隐患已全部排除！")
print("⚠️ 【极度重要】：底层显卡库已更新，请立刻点击顶部菜单栏：")
print("👉 [代码执行程序 (Runtime)] -> [重新启动会话 (Restart session)]")
print("============================================================")

In [ ]:
# ==========================================
# Cell 3: 高速静默下载 FishAudio s2-pro 大模型
# ==========================================
import os
print("📥 正在开启多线程满速下载 fishaudio/s2-pro 官方大模型...")
print("⏳ 这是一个 5B 参数的大模型，需要几分钟时间，请耐心等待...")

# 开启高速传输协议
!pip install -q -U huggingface_hub hf_transfer
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import snapshot_download

# 精准定位到 FishAudio 插件识别的目录
model_dir = "/content/ComfyUI/models/fishaudioS2/s2-pro"
os.makedirs(model_dir, exist_ok=True)

try:
    # 直接拉取 s2-pro 核心大模型，并强制保存为实体文件
    path = snapshot_download(
        repo_id="fishaudio/s2-pro",
        local_dir=model_dir,
        local_dir_use_symlinks=False
    )
    print(f"\n✅ Cell 3 完成！s2-pro 大模型已完美落盘到本地！")
except Exception as e:
    print(f"\n❌ 下载出现意外: {e}")

In [ ]:
# ==========================================
# Cell 4: 启动专属服务器 (TCP直连 + 防断开保活)
# ==========================================
import os, time, threading, subprocess, configparser
from google.colab import userdata

%cd /content/ComfyUI

# ---------------------------------------------------------
# 1. 下载 FRP 并配置 TCP 穿透
# ---------------------------------------------------------
frp_dir = "/content/frp_0.56.0_linux_amd64"
if not os.path.exists(frp_dir):
    os.system("wget -q https://github.com/fatedier/frp/releases/download/v0.56.0/frp_0.56.0_linux_amd64.tar.gz -O /content/frp.tar.gz")
    os.system("tar -xzf /content/frp.tar.gz -C /content/")
    os.system("rm /content/frp.tar.gz")
    os.system(f"chmod +x {frp_dir}/frpc")

try:
    vps_ip = userdata.get('VPS_IP')
    frp_token = userdata.get('FRP_TOKEN')
    toml_content = f"""
serverAddr = "{vps_ip}"
serverPort = 7000
auth.method = "token"
auth.token = "{frp_token}"

[[proxies]]
name = "comfyui_cjp_tcp"
type = "tcp"
localIp = "127.0.0.1"
localPort = 8188
remotePort = 8086
"""
    with open(f"{frp_dir}/frpc.toml", "w") as f:
        f.write(toml_content.strip())
except Exception as e:
    print(f"❌ 读取 Secrets 失败！详情: {e}")

# ---------------------------------------------------------
# 2. 启动 FRP 与 防断开保活
# ---------------------------------------------------------
def start_frpc():
    process = subprocess.Popen([f"{frp_dir}/frpc", "-c", f"{frp_dir}/frpc.toml"], stdout=subprocess.PIPE, text=True)
    for i in range(5):
        line = process.stdout.readline()
        if not line: break
        print(f"[FRP] {line.strip()}")

def keep_alive():
    while True:
        time.sleep(300)
        print("\n[Keep-Alive] 保持 Colab 连接活跃中...")

threading.Thread(target=start_frpc, daemon=True).start()
threading.Thread(target=keep_alive, daemon=True).start()

print("\n============================================================")
print("👉 启动完成后，请访问: http://cjp.usdream.dpdns.org:8086")
print("============================================================\n")

# ---------------------------------------------------------
# 3. 屏蔽可能导致卡死的 Manager 联网动作，并启动
# ---------------------------------------------------------
manager_paths = ["/content/ComfyUI/user/__manager/config.ini", "/content/ComfyUI/custom_nodes/ComfyUI-Manager/config.ini"]
for cp in manager_paths:
    os.makedirs(os.path.dirname(cp), exist_ok=True)
    config = configparser.ConfigParser()
    if os.path.exists(cp): config.read(cp)
    if 'default' not in config: config['default'] = {}
    config['default']['network_mode'] = 'private'
    with open(cp, 'w') as f: config.write(f)

!python main.py --dont-print-server